# 6. Global minimum

In [1]:
import numpy as np 
import matplotlib.pyplot as plt

$$f(z) = -\sum_{i=1}^4 c_i \exp\left(-\sum_{j=1}^3 a_{ij}(z_j - p_{ij})^2\right)$$

Partial derivateive is:
$$\frac{\partial f}{\partial z_k} = -\sum_{i=1}^4 c_i \exp\left(-\sum_{j=1}^3 a_{ij}(z_j - p_{ij})^2\right) \cdot \left(-2a_{ik}(z_k - p_{ik})\right)$$

Cancelling negative signs:
$$\frac{\partial f}{\partial z_k} = \sum_{i=1}^4 2 c_i a_{ik} (z_k - p_{ik}) \exp\left(-\sum_{j=1}^3 a_{ij}(z_j - p_{ij})^2\right)$$

In [3]:
#defining the table contents

C = np.array([1.0, 1.2, 3.0, 3.2])

A = np.array([
    [3.0, 10.0, 30.0],
    [0.1, 10.0, 35.0],
    [3.0, 10.0, 30.0],
    [0.1, 10.0, 35.0]
])

P = np.array([
    [0.36890, 0.1170, 0.2673],
    [0.46990, 0.4387, 0.7470],
    [0.10910, 0.8732, 0.5547],
    [0.03815, 0.5743, 0.8828]
])

**OBJECTIVE AND GRADIENT FUNCTIONS**

In [4]:
def f(z):
    """Calculates function value"""
    total = 0.0
    for i in range(4):
        # Calculate the inner sum for the exponent
        inner_sum = np.sum(A[i] * (z - P[i])**2)
        total -= C[i] * np.exp(-inner_sum)
    return total

In [6]:
def grad_f(z):
    """Calculates the gradient of function"""
    grad = np.zeros(3)
    for i in range(4):
        inner_sum = np.sum(A[i] * (z - P[i])**2)
        exp_term = np.exp(-inner_sum)
        # The gradient component formula derived at the start
        for k in range(3):
            grad[k] += 2 * C[i] * A[i, k] * (z[k] - P[i, k]) * exp_term
    return grad

In [7]:
def proj_box(z):
    """Projects z back to [0,1]^3"""
    return np.clip(z, 0.0, 1.0)

**MULTI-START PGD**

**What should the learning rate be?**
Since the curvature of the function is roughly dictated by the formula $2 \cdot c_i \cdot a_{ij}$ near the bottom of the valleys, we must look at the highest value - which is:
* $c_4 = 3.2$
* $a_{4,3} = 35$

Then: $\beta \approx 2 \cdot 3.2 \cdot 35 = 224$

So the LR should be: $\gamma_{safe} = \frac{1}{224} \approx 0.0044$

In [18]:
def pgd(init_z, lr, max_steps=3000, tolerance=1e-8):
    """Runs PGD until convergence or max_steps"""
    z = np.array(init_z, dtype=float)
    
    for step in range(1, max_steps + 1):
        gradient = grad_f(z)
        z_new = proj_box(z - lr * gradient)
        
        # Stop early if the algorithm barely moves (convergence)
        if np.linalg.norm(z_new - z) < tolerance:
            return z_new, f(z_new), step
            
        z = z_new
        
    return z, f(z), max_steps

In [19]:
np.random.seed(37) # For reproducible random starts
num_starts = 100
learning_rate = 0.003 #safely below 0.0044

best_z = None
best_f = float('inf')
best_steps = 0

print(f"{'Run':<5} | {'Start point':<30} | {'Final value':<15} | {'steps'}")
print("-" * 65)

for i in range(num_starts):
    # Drop a random point in [0,1]^3
    start_z = np.random.rand(3)
    
    #run PGD
    final_z, final_f, steps_taken = pgd(start_z, lr=learning_rate)
    
    #Track the global best
    if final_f < best_f:
        best_f = final_f
        best_z = final_z
        best_steps = steps_taken
        
    if i < 5 or i == num_starts - 1: # print a few 
        print(f"{i+1:<5} | {np.round(start_z, 3)} | {final_f:<15.6f} | {steps_taken}")

print("-" * 65)
print(f"\nGLOBAL BEST FOUND:")
print(f"Coordinates : {np.round(best_z, 5)}")
print(f"Minimum val : {best_f:.14f}")
print(f"Steps taken : {best_steps}")
print(f"Target val  : -3.86278214782076")
print(f"Difference  : {abs(best_f - -3.86278214782076):.14e}")

Run   | Start point                    | Final value     | steps
-----------------------------------------------------------------
1     | [0.944 0.464 0.193] | -1.000817       | 836
2     | [0.582 0.62  0.684] | -3.862782       | 3000
3     | [0.103 0.745 0.282] | -3.089764       | 193
4     | [0.753 0.793 0.627] | -3.862782       | 3000
5     | [0.443 0.963 0.897] | -3.862782       | 3000
100   | [0.332 0.01  0.434] | -1.000817       | 616
-----------------------------------------------------------------

GLOBAL BEST FOUND:
Coordinates : [0.11462 0.55565 0.85255]
Minimum val : -3.86278214781623
Steps taken : 2958
Target val  : -3.86278214782076
Difference  : 4.52970994047064e-12
